In [1]:
!uv add pandas

Resolved 40 packages in 26.59s
Prepared 2 packages in 2m 01s
Installed 4 packages in 587ms
 + numpy==2.3.3
 + pandas==2.3.2
 + pytz==2025.2
 + tzdata==2025.2


In [1]:
from concurrent.futures import ProcessPoolExecutor as ppe
from generate import generate_csv

NUMBER_OF_FILES = 5
NUMBER_OF_LINES = 1000

if __name__ == "__main__":
    with ppe() as executor:
        futures = [executor.submit(generate_csv,i,NUMBER_OF_LINES) for i in range(1,NUMBER_OF_FILES + 1)]

fn = []
for future in futures:
    fn.append(future.result())
    
futures

[<Future at 0x1d017678a50 state=finished returned str>,
 <Future at 0x1d017678e10 state=finished returned str>,
 <Future at 0x1d0175f9810 state=finished returned str>,
 <Future at 0x1d0175f9a70 state=finished returned str>,
 <Future at 0x1d01760a330 state=finished returned str>]

In [2]:
import pandas as pd
from concurrent.futures import ThreadPoolExecutor as tpe

def process(path):
    df = pd.read_csv(path)
    return df.groupby('Категория')['Значение'].agg(Медиана = 'median', Стандартное_отклонение='std').reset_index()

with tpe() as executor:
    dataframes = list(executor.map(process,fn))

combined = pd.concat(dataframes, ignore_index=True)

combined

,Категория,Медиана,Стандартное_отклонение
0,A,4502.054235,2936.560917
1,B,5198.779573,2915.918010
2,C,5477.606771,2957.103800
3,D,4503.866412,2767.250483
4,A,4802.699381,3034.352699
5,B,4773.230552,2741.172212
6,C,4894.889379,2859.049331
7,D,5187.042723,2840.736113
8,A,5108.409643,2736.428579
9,B,5147.958567,2865.302599


In [3]:
result = combined.groupby('Категория')['Медиана'].agg(Медиана_из_медиан='median', Стандартное_отклонение_из_медиан='std').reset_index()
result

,Категория,Медиана_из_медиан,Стандартное_отклонение_из_медиан
0,A,4802.699381,232.327913
1,B,5147.958567,214.596874
2,C,4894.889379,395.185769
3,D,5140.220188,304.896737
